# Load all three layers into DuckDB

In [14]:
import duckdb

con = duckdb.connect(
    r"D:\pyspark_vs_code\medallion_project_duckdb\database\sales_db.duckdb"
)

# Create schemas
con.sql("CREATE SCHEMA IF NOT EXISTS bronze")
con.sql("CREATE SCHEMA IF NOT EXISTS silver")
con.sql("CREATE SCHEMA IF NOT EXISTS gold")

CountBIGINT


In [17]:
con.sql("show schemas")
con.sql(" use bronze")
con.sql("show tables")

nameVARCHAR
raw_sales


# Load the Parquet datasets into permanent DuckDB tables:

In [8]:
base_path = "D:/pyspark_vs_code/medallion_project_duckdb/data"

for layer, table in [
    ("bronze", "raw_sales"),
    ("silver", "clean_sales"),
    ("gold", "state_sales")
]:
    con.sql(f"""
        CREATE OR REPLACE TABLE {layer}.{table} AS
        SELECT *
        FROM read_parquet(
            '{base_path}/{layer}/*.parquet'
        )
    """)

# query tables using three-part names:

In [9]:
con.sql("""
    SELECT *
    FROM sales_db.gold.state_sales
    ORDER BY total_sales DESC
""").show()

┌──────────────────────┬───────────────┬──────────────┐
│        state         │  total_sales  │ total_orders │
│       varchar        │ decimal(23,2) │    int64     │
├──────────────────────┼───────────────┼──────────────┤
│ California           │    1334534.23 │         7268 │
│ Texas                │    1039762.89 │         6406 │
│ Florida              │     532213.01 │         4095 │
│ Oregon               │     366316.69 │         3005 │
│ Ohio                 │     359566.70 │         2981 │
│ Colorado             │     355114.52 │         2919 │
│ Tennessee            │     352222.28 │         2938 │
│ Washington           │     345998.70 │         2947 │
│ Virginia             │     268891.32 │         2256 │
│ Massachusetts        │     267025.67 │         2268 │
│  ·                   │         ·     │           ·  │
│  ·                   │         ·     │           ·  │
│  ·                   │         ·     │           ·  │
│ Utah                 │     168689.40 │        

In [10]:
con.sql("""
    SELECT *
    FROM gold.state_sales
""").show()

┌────────────────┬───────────────┬──────────────┐
│     state      │  total_sales  │ total_orders │
│    varchar     │ decimal(23,2) │    int64     │
├────────────────┼───────────────┼──────────────┤
│ Utah           │     168689.40 │         1538 │
│ Hawaii         │      85340.69 │          814 │
│ Minnesota      │     181497.34 │         1590 │
│ Ohio           │     359566.70 │         2981 │
│ Oregon         │     366316.69 │         3005 │
│ Texas          │    1039762.89 │         6406 │
│ Pennsylvania   │     178550.70 │         1604 │
│ Connecticut    │     177010.04 │         1599 │
│ Nebraska       │     177600.32 │         1601 │
│ Nevada         │     182117.16 │         1620 │
│   ·            │          ·    │           ·  │
│   ·            │          ·    │           ·  │
│   ·            │          ·    │           ·  │
│ Mississippi    │      90551.17 │          840 │
│ Tennessee      │     352222.28 │         2938 │
│ Florida        │     532213.01 │         4095 │


In [11]:
con.sql("""
    SELECT
        table_schema,
        table_name
    FROM information_schema.tables
    WHERE table_schema IN ('bronze', 'silver', 'gold')
    ORDER BY table_schema, table_name
""").show()

┌──────────────┬─────────────┐
│ table_schema │ table_name  │
│   varchar    │   varchar   │
├──────────────┼─────────────┤
│ bronze       │ raw_sales   │
│ gold         │ state_sales │
│ silver       │ clean_sales │
└──────────────┴─────────────┘



# compare record counts:

In [7]:
con.sql("""
    SELECT 'bronze' AS layer, COUNT(*) AS records
    FROM bronze.raw_sales

    UNION ALL

    SELECT 'silver', COUNT(*)
    FROM silver.clean_sales

    UNION ALL

    SELECT 'gold', COUNT(*)
    FROM gold.state_sales
""").show()

┌─────────┬─────────┐
│  layer  │ records │
│ varchar │  int64  │
├─────────┼─────────┤
│ bronze  │   95904 │
│ silver  │   95904 │
│ gold    │      38 │
└─────────┴─────────┘



In [13]:
con.close()